# Hands-On Assignment 5

In this assignment, you will practice everything that you have learned so far in an end-to-end setting.
You will be provided with a dataset that is **unique to you**, and your task is to perform
all the steps from previous assignments to clean, explore, visualize, and analyze your dataset.

**Written Portion**: Additionally, you will create a report that describes your process and provides insights about your dataset.
Each section that should appear in your report is noted with an orange star (like normal HO tasks).  The report should be  4-6 pages (12 pt font, 1.5 line spacing), and turned in on Canvas as a PDF.

The coding aspect for this assignment will be turned in the same was as all other HO's,
by submitting this file to the autograder.


For this assignment, feel free to make additional functions instead of implementing everything in the provided function.

The objective of this assignment is for you to apply and solidify the skills you have learned in previous assignments.

# Prompt

You have graduated from this class, and are a huge success!
You landed a job doing data science at some fancy company.

You just got a new client with some really interesting problems you get to solve.
Unfortunately, because of a big mess-up on their side the data's metadata got corrupted
(and the person that used to maintain the data just took a vow of silence and moved to a bog).

The only column you are sure about is the `label` column,
which contains a numeric label for each row.
Aside from that, the client does not know anything about the names, content, or even data types for each column.

Your task is to explore, clean, and analyze this data.
You should have already received an email with the details on obtaining your unique data.
Place it in the same directory as this notebook (and your `local_grader.py` script) and name it `data.txt`.

*I know this prompt may sound unrealistic, but I have literally been in a situation exactly like this.
I was working at a database startup, and one of our clients gave us data with over 70 columns and more than a million records and told us:
"The person who used to manage the data is no longer working with us, but this was the data they used to make all their decisions.
We also lost all the metadata information, like column names."
...
Working in industry is not always glamorous.
-Eriq*

# Part 0: Explore Your Data

Before you start doing things to/with your data, it's always a good idea to load up your data and take a look.

In [16]:
import pandas
import sklearn.linear_model
import sklearn.neighbors
import sklearn.tree
import numpy
import scipy

# Modify this to point to your data.
unique_data = pandas.read_csv('data.txt', sep = "\t")
unique_data

,label,col_00,col_01,col_02,col_03,col_04,col_05,col_06,col_07,col_08,col_09
0,4,237 kg,0.0913 N,SOCCER,ELECTRICAL ENGINEERING,-630,1043.0,long beach,1163 m^2,0.881,1187.0
1,1,2157 kg,0.146 N,volleyball,"Bioinformatics, Human Computer Interaction",758,520.0,Los Angeles,606 m^2,-1.484,-711.0
2,2,1271 kg,0.5166 N,track & field,"Data Science, Computer Science",648,325.0,Irvine,2040 m^2,-0.1644,15.0
3,4,2138 kg,1.6769 N,soccer,Electrical Engineering,1280,531.0,San Bernardino,55 m^2,0.7802,1196.0
4,2,1520 kg,0.7643 N,track & field,Technology and Information Management,1533,-172.0,"Riverside, Chula Vista",659 m^2,-0.0208,1642.0
...,...,...,...,...,...,...,...,...,...,...,...
1246,4,-373 kg,0.6918 N,ice hockey,Electrical Engineering,3086,935.0,San Bernardino,-216 m^2,1.4806,-517.0
1247,0,934 kg,0.6265 N,baseball,Biotechnology,-281,-668.0,"Oakland, Modesto, San Diego",415 m^2,1.1251,-210.0
1248,0,1166 kg,-0.7745 N,baseball,"Computer Engineering, Robotics Engineering",-854,618.0,Modesto,-45 m^2,0.8299,675.0
1249,1,1507 kg,1.7912 N,badminton,"bioinformatics, human computer interaction",-61,-333.0,"Moreno Valley, Bakersfield, Riverside, Chula V...",806 m^2,-0.7481,338.0


Don't forget to checkout the column information.

In [3]:
unique_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1251 entries, 0 to 1250
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   label   1251 non-null   int64  
 1   col_00  1242 non-null   object 
 2   col_01  1232 non-null   object 
 3   col_02  1239 non-null   object 
 4   col_03  1245 non-null   object 
 5   col_04  1236 non-null   object 
 6   col_05  1245 non-null   float64
 7   col_06  1242 non-null   object 
 8   col_07  1237 non-null   object 
 9   col_08  1243 non-null   object 
 10  col_09  1239 non-null   float64
dtypes: float64(2), int64(1), object(8)
memory usage: 107.6+ KB


And any numeric information.

In [4]:
unique_data.describe()

,label,col_05,col_09
count,1251.000000,1245.000000,1239.000000
mean,1.988809,755.726908,578.991929
std,1.408501,1043.789679,946.460526
min,0.000000,-2646.000000,-2483.000000
25%,1.000000,50.000000,-49.000000
50%,2.000000,743.000000,608.000000
75%,3.000000,1407.000000,1199.500000
max,4.000000,4631.000000,3399.000000


<h4 style="color: darkorange; font-size: x-large";>★ Written Task: Introduction</h4>

Briefly describe the dataset you’re given and define the goal of the project and how you approach it.
For example, you can present a basic introduction of your data (shape and proposed data types)
and your goal is to use these features to predict the label of the response variable.
Then you propose a few models that are suitable for this project which will be introduced in the modeling section.

# Part 1: Data Cleaning

As always, we should start with data cleaning.
Take what you learned from HO3 to clean up this messy data to a point where it is ready for machine learning algorithms.

Some things you may want to do:
 - Deal with missing/empty values.
 - Fix numeric columns so that they actually contain numbers.
 - Remove inconsistencies from columns.
 - Assign a data type to each column.

<h4 style="color: darkorange; font-size: x-large";>★ Task 1.A</h4>

Complete the following function that takes in a DataFrame and outputs a clean version of the DataFrame.
You can assume that the frame has all the same structure as your unique dataset.
You can return the same or a new data frame.

In [18]:
def clean_data(frame):
    
    df = frame.copy()

    # normalize missing values
    missing_vals = ["?", "None", "NULL", "n/a", "N/A"]
    df.replace(missing_vals, numpy.nan, inplace=True)

    # remove units from numeric columns
    df["col_00"] = df["col_00"].str.replace(" kg", "", regex=False)
    df["col_01"] = df["col_01"].str.replace(" N", "", regex=False)
    df["col_07"] = df["col_07"].str.replace(" m^2", "", regex=False)

    # convert numeric columns
    numeric_cols = ["col_00", "col_01", "col_04", "col_05", "col_07", "col_08", "col_09"]
    for col in numeric_cols:
        df[col] = pandas.to_numeric(df[col], errors="coerce")

    # standardize categorical text
    df["col_02"] = df["col_02"].str.lower()
    df["col_03"] = df["col_03"].str.lower()
    df["col_06"] = df["col_06"].str.lower()

    # fill missing numeric values with column means
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].mean())

    return df


unique_data = clean_data(unique_data)
unique_data

,label,col_00,col_01,col_02,col_03,col_04,col_05,col_06,col_07,col_08,col_09
0,4,237.0,0.0913,soccer,electrical engineering,-630.0,1043.0,long beach,1163.0,0.8810,1187.0
1,1,2157.0,0.1460,volleyball,"bioinformatics, human computer interaction",758.0,520.0,los angeles,606.0,-1.4840,-711.0
2,2,1271.0,0.5166,track & field,"data science, computer science",648.0,325.0,irvine,2040.0,-0.1644,15.0
3,4,2138.0,1.6769,soccer,electrical engineering,1280.0,531.0,san bernardino,55.0,0.7802,1196.0
4,2,1520.0,0.7643,track & field,technology and information management,1533.0,-172.0,"riverside, chula vista",659.0,-0.0208,1642.0
...,...,...,...,...,...,...,...,...,...,...,...
1246,4,-373.0,0.6918,ice hockey,electrical engineering,3086.0,935.0,san bernardino,-216.0,1.4806,-517.0
1247,0,934.0,0.6265,baseball,biotechnology,-281.0,-668.0,"oakland, modesto, san diego",415.0,1.1251,-210.0
1248,0,1166.0,-0.7745,baseball,"computer engineering, robotics engineering",-854.0,618.0,modesto,-45.0,0.8299,675.0
1249,1,1507.0,1.7912,badminton,"bioinformatics, human computer interaction",-61.0,-333.0,"moreno valley, bakersfield, riverside, chula v...",806.0,-0.7481,338.0


Now we should also be able to view all the numeric columns.

In [7]:
unique_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1251 entries, 0 to 1250
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   label   1251 non-null   int64  
 1   col_00  1251 non-null   float64
 2   col_01  1251 non-null   float64
 3   col_02  1236 non-null   object 
 4   col_03  1242 non-null   object 
 5   col_04  1251 non-null   float64
 6   col_05  1251 non-null   float64
 7   col_06  1241 non-null   object 
 8   col_07  1251 non-null   float64
 9   col_08  1251 non-null   float64
 10  col_09  1251 non-null   float64
dtypes: float64(7), int64(1), object(3)
memory usage: 107.6+ KB


<h4 style="color: darkorange; font-size: x-large";>★ Written Task: Data Cleaning</h4>

Describe the steps you took for data cleaning.
Why did you do this?
Did you have to make some choices along the way? If so, describe them.

# Part 2: Data Visualization

Once you have cleaned up the data, it is time to explore it and find interesting things.
Part of this exploration, will be visualizing the data in a way that makes it easier for yourself and others to understand.
Use what you have learned in HO1 and HO2 to create some visualizations for your dataset.

<h4 style="color: darkorange; font-size: x-large";>★ Written Task: Data Visualization</h4>

Create at least two different visualizations that help describe what you see in your dataset.
Include these visualizations in your report along with descriptions of
how you created the visualization,
what data preparation you had to do for the visualization (aside from the data cleaning in the previous part),
and what the visualization tells us about the data.

# Part 3: Modeling

Now that you have a good grasp of your clean data,
it is time to do some machine learning!
(Technically all our previous steps were also machine learning,
but now we get to use classifiers!)

Use the skills you developed to select **three** classifiers and implement them on your data.
For example, you can narrow down your choices to three classifiers which may include:
- Logistic regression
- K-nearest neighbors
- Decision tree
- Or others

<h4 style="color: darkorange; font-size: x-large";>★ Task 3.A</h4>

Complete the following function that takes in no parameters,
and returns a list with **three** untrained classifiers you are going to explore in this assignment.
This method may set parameters/options for the classifiers, but should not do any training/fitting.

For example, if you wanted to use logistic regression,
then **one** of your list items may be:
```
sklearn.linear_model.LogisticRegression()
```

In [ ]:
def create_classifiers():

    clf1 = sklearn.tree.DecisionTreeClassifier()
    clf2 = sklearn.neighbors.KNeighborsClassifier()
    clf3 = sklearn.linear_model.LogisticRegression()

    return [clf1, clf2, clf3]

my_classifiers = create_classifiers()
my_classifiers

[DecisionTreeClassifier(), KNeighborsClassifier(), LogisticRegression()]

Now that we have some classifiers, we can see how they perform.

<h4 style="color: darkorange; font-size: x-large";>★ Task 3.B</h4>

Complete the following function that takes in an untrained classifier, a DataFrame, and a number of folds.
This function should run k-fold cross validation with the classifier and the data,
and return a list with the accuracy of each run of cross validation.
You can assume that the frame has the column `label` and the rest of the columns can be considered clean numeric features.

Note that you may have to break your frame into features and labels to do this.
Do not change the passed-in frame (make copies instead).

If you are getting any `ConvergenceWarning`s you may either ignore them,
or try and address them
(they will not affect your autograder score, but may be something to discuss in the written portion of this assignment).

In [14]:
def cross_fold_validation(classifier, frame, folds):

    data = frame.copy()

    # separate labels
    y = data["label"]

    # features
    X = data.drop(columns=["label"])

    # convert categorical columns to numeric, used GeeksforGeeks as reference:
    # https://www.geeksforgeeks.org/pandas/python-pandas-get_dummies-method/
    X = pandas.get_dummies(X)

    kf = sklearn.model_selection.KFold(n_splits=folds, shuffle=True)

    scores = []

    for train_index, test_index in kf.split(X):

        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        model = sklearn.base.clone(classifier)
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        accuracy = sklearn.metrics.accuracy_score(y_test, predictions)

        scores.append(accuracy)

    return scores

my_classifiers_scores = []
for classifier in my_classifiers:
    accuracy_scores = cross_fold_validation(classifier, unique_data, 5)
    my_classifiers_scores.append(accuracy_scores)
    print("Classifier: %s, Accuracy: %s." % (type(classifier).__name__, accuracy_scores))

Classifier: DecisionTreeClassifier, Accuracy: [0.900398406374502, 0.884, 0.892, 0.868, 0.86].
Classifier: KNeighborsClassifier, Accuracy: [0.23904382470119523, 0.264, 0.292, 0.248, 0.268].


/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:

Classifier: LogisticRegression, Accuracy: [0.9402390438247012, 0.912, 0.92, 0.904, 0.932].


/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/nikop/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


<h4 style="color: darkorange; font-size: x-large";>★ Task 3.C</h4>

Complete the following function that takes in two equally-sized lists of numbers and a p-value.
This function should compute whether there is a statistical significance between
these two lists of numbers using a [Student's t-test](https://en.wikipedia.org/wiki/Student%27s_t-test)
at the given p-value.
Return `True` if there is a statistical significance, and `False` otherwise.
Hint: If you wish, you may use the `ttest_ind()` [method](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html) provided in the scipy package. 

In [17]:
def significance_test(a_values, b_values, p_value):

    t_stat, p = scipy.stats.ttest_ind(a_values, b_values)

    return p < p_value

for i in range(len(my_classifiers)):
    for j in range(i + 1, len(my_classifiers)):
        significant = significance_test(my_classifiers_scores[i], my_classifiers_scores[j], 0.10)
        print("%s vs %s: %s" % (type(my_classifiers[i]).__name__,
                                type(my_classifiers[j]).__name__, significant))

DecisionTreeClassifier vs KNeighborsClassifier: True
DecisionTreeClassifier vs LogisticRegression: True
KNeighborsClassifier vs LogisticRegression: True


<h4 style="color: darkorange; font-size: x-large";>★ Written Task: Modeling</h4>

Describe the classifiers you have chosen.
Be sure to include all details about any parameter settings used for the algorithms.

Compare the performance of your models using k-fold validation.
You may look at accuracy, F1 or other measures.

Then, briefly summarize your results.
Are your results statistically significant?
Is there a clear winner?
What do the standard deviations look like, and what do they tell us about the different models?
Include a table like Table 1.

<center>Table 1: Every table needs a caption.</center>

| Model | Mean Accuracy | Standard Deviation of Accuracy |
|-------|---------------|--------------------------------|
| Logistic Regression | 0.724 | 0.004
| K-Nearest Neighbor | 0.750 | 0.003
| Decision Tree | 0.655 | 0.011

# Part 4: Analysis

Now, take some time to go over your results for each classifier and try to make sense of them.
 - Why do some classifiers work better than others?
 - Would another evaluation metric work better than vanilla accuracy?
 - Is there still a problem in the data that should be fixed in data cleaning?
 - Does the statistical significance between the different classifiers make sense?
 - Are there parameters for the classifier that I can tweak to get better performance?

<h4 style="color: darkorange; font-size: x-large";>★ Written Task: Analysis</h4>

Discuss your observations, the relationship you found, and how you applied concepts from the class to this project.
For example, you may find that some feature has the most impact in predicting your response variable or removing a feature improves the model accuracy.
Or you may observe that your training accuracy is much higher than your test accuracy and you may want to explain what issues may arise.

# Part 5: Conclusion

<h4 style="color: darkorange; font-size: x-large";>★ Written Task: Conclusion</h4>

Briefly summarize the important results and conclusions presented in the project.
What are the important points illustrated by your work?
Are there any areas for further investigation or improvement?

<h4 style="color: darkorange; font-size: x-large";>★ Written Task: References</h4>

Include a standard bibliography with citations referring to techniques or published papers you used throughout your report (if you used any).

For example:
```
[1] Derpanopoulos, G. (n.d.). Bayesian Model Checking & Comparison.
https://georgederpa.github.io/teaching/modelChecking.html.
```

# Part XC: Extra Credit

So far you have used a synthetic dataset that was created just for you.
But, data science is always more interesting when you are dealing with actual data from the real world.
Therefore, you will have an opportunity for extra credit on this assignment using real-world data.

For extra credit, repeat the **written tasks** of Parts 0 through 4 with an additional dataset that you find yourself.
For the written portion of the extra credit for Part 0, include information about where you got the data and what the data represents.
You may choose any dataset that represents real data (i.e., is **not** synthetic or generated)
and is **not** [pre-packaged in scikit-learn](https://scikit-learn.org/stable/datasets.html).

Below are some of the many places you can start looking for datasets:
 - [Kaggle](https://www.kaggle.com/datasets) -- Kaggle is a website focused around machine learning competitions,
       where people compete to see who can get the best results on a dataset.
       It is very popular in the machine learning community and has thousands of datasets with descriptions.
       Make sure to read the dataset's description, as Kaggle also has synthetic datasets.
 - [data.gov](https://data.gov/) -- A portal for data from the US government.
        The US government has a lot of data, and much of it has to be available to the public by law.
        This portal contains some of the more organized data from several different government agencies.
        In general, the government has A LOT of interesting data.
        It may not always be clean (remember the CIA factbook), but it is interesting and available.
        All data here should be real-world, but make sure to read the description to verify.
 - [UCI's Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets.php) -- UC Irvine has their own data repository with a few hundred datasets on many different topics.
        Make sure to read the dataset's description, as UCI also has synthetic datasets.
 - [WHO's Global Health Observatory](https://apps.who.int/gho/data/node.home) -- The World Health Organization keeps track of many different health-related statistics for most of the countries in the world.
        All data here should be real-world, but make sure to read the description to verify.
 - [Google's Dataset Search](https://datasetsearch.research.google.com/) -- Google indexes many datasets that can be searched here.

You can even create a dataset from scratch if you find some data you like that is not already organized into a specific dataset.
The only real distinction between "data" and a "dataset" is that a dataset is organized and finite (has a fixed size).

Create a new section in your written report for this extra credit and include all the written tasks for the extra credit there.
Each written task/section that you complete for your new dataset is eligible for extra credit (so you can still receive some extra credit even if you do not complete all parts).
There is no need to submit any code for the extra credit.
If you created a new dataset, include the dataset or links to it with your submission.